# MLB Odds & Predictions Analysis
Interactive notebook for exploring historical predictions and odds data.

In [ ]:
import sqlite3
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

DB_PATH = Path('backend/data/predictions.db')
con = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)

## 1. What's in the database?

In [ ]:
# High-level counts
summary = pd.read_sql("""
    SELECT
        COUNT(*)                                          AS total_games,
        COUNT(DISTINCT game_date)                         AS unique_dates,
        MIN(game_date)                                    AS earliest,
        MAX(game_date)                                    AS latest,
        SUM(resolved)                                     AS resolved,
        COUNT(*) - SUM(resolved)                          AS pending,
        SUM(is_value_pick)                                AS value_picks,
        COUNT(DISTINCT model_used)                        AS models_used
    FROM predictions
""", con)
summary.T

In [ ]:
# Backfill log — credits used per day
pd.read_sql("""
    SELECT date, events_fetched, credits_used, credits_remaining
    FROM backfill_log
    ORDER BY date
""", con)

## 2. Load predictions

In [ ]:
df = pd.read_sql("""
    SELECT
        id, game_date, home_team, away_team,
        predicted_winner, home_win_prob, away_win_prob,
        is_value_pick, home_best_odds, away_best_odds,
        model_used, actual_winner, resolved
    FROM predictions
    ORDER BY game_date
""", con)

df['game_date'] = pd.to_datetime(df['game_date'])
df['correct']   = (df['predicted_winner'] == df['actual_winner']).where(df['resolved'] == 1)

# Win probability of the predicted team
df['pred_prob'] = np.where(
    df['predicted_winner'] == df['home_team'],
    df['home_win_prob'], df['away_win_prob']
)
# Best odds for the predicted team
df['pred_odds'] = np.where(
    df['predicted_winner'] == df['home_team'],
    df['home_best_odds'], df['away_best_odds']
)

print(f'{len(df)} games  |  {df["resolved"].sum()} resolved  |  {df["is_value_pick"].sum()} value picks')
df.head()

## 3. Performance overview

In [ ]:
resolved = df[df['resolved'] == 1].copy()

def perf(subset, label):
    if len(subset) == 0:
        return {'label': label, 'games': 0}
    acc = subset['correct'].mean()
    return {'label': label, 'games': len(subset),
            'correct': int(subset['correct'].sum()),
            'accuracy': round(acc, 4)}

rows = [
    perf(resolved, 'Overall'),
    perf(resolved[resolved['is_value_pick'] == 1], 'Value picks'),
    perf(resolved[resolved['is_value_pick'] == 0], 'Non-value picks'),
]
for m in resolved['model_used'].unique():
    rows.append(perf(resolved[resolved['model_used'] == m], f'Model: {m}'))

pd.DataFrame(rows).set_index('label')

In [ ]:
# Accuracy over time (7-day rolling)
daily = resolved.groupby('game_date')['correct'].mean().reset_index()
daily['rolling_7'] = daily['correct'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(daily['game_date'], daily['correct'], alpha=0.3, color='steelblue', label='Daily')
ax.plot(daily['game_date'], daily['rolling_7'], color='steelblue', lw=2, label='7-day avg')
ax.axhline(0.5, color='red', lw=1, ls='--', label='50%')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title('Prediction accuracy over time')
ax.legend()
plt.tight_layout()
plt.show()

## 4. ROI simulation

In [ ]:
def american_to_payout(odds):
    """Profit on a $100 bet."""
    if pd.isna(odds):
        return np.nan
    return 100 / abs(odds) * 100 if odds < 0 else odds / 100 * 100

sim = resolved.dropna(subset=['pred_odds']).copy()
sim['profit'] = sim.apply(
    lambda r: american_to_payout(r['pred_odds']) if r['correct'] else -100, axis=1
)
sim['cumulative_profit'] = sim.sort_values('game_date')['profit'].cumsum()

# Overall vs value picks
vp = sim[sim['is_value_pick'] == 1].copy()
vp['cumulative_profit'] = vp.sort_values('game_date')['profit'].cumsum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sim.sort_values('game_date')['game_date'],
        sim.sort_values('game_date')['cumulative_profit'],
        label='All picks', color='steelblue')
if len(vp):
    ax.plot(vp.sort_values('game_date')['game_date'],
            vp.sort_values('game_date')['cumulative_profit'],
            label='Value picks only', color='green', lw=2)
ax.axhline(0, color='red', lw=1, ls='--')
ax.set_ylabel('Cumulative profit ($100/game)')
ax.set_title('Simulated ROI')
ax.legend()
plt.tight_layout()
plt.show()

total = len(sim) * 100
print(f'All picks    — wagered ${total:,.0f}  profit ${sim["profit"].sum():,.0f}  ROI {sim["profit"].sum()/total:.1%}')
if len(vp):
    vt = len(vp) * 100
    print(f'Value picks  — wagered ${vt:,.0f}  profit ${vp["profit"].sum():,.0f}  ROI {vp["profit"].sum()/vt:.1%}')

## 5. Calibration — does 60% actually win 60% of the time?

In [ ]:
cal = resolved.copy()
cal['bucket'] = (cal['pred_prob'] * 10).astype(int) / 10
cal = cal.groupby('bucket').agg(
    games=('correct', 'count'),
    actual_win_rate=('correct', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(cal['bucket'], cal['actual_win_rate'], s=cal['games']*4, color='steelblue', zorder=3)
ax.plot([0.5, 1], [0.5, 1], 'r--', lw=1, label='Perfect calibration')
for _, row in cal.iterrows():
    ax.annotate(f"n={int(row['games'])}", (row['bucket'], row['actual_win_rate']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set_xlabel('Model confidence')
ax.set_ylabel('Actual win rate')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title('Calibration plot')
ax.legend()
plt.tight_layout()
plt.show()

cal

## 6. Odds deep-dive — parse per-book lines

In [ ]:
# Load full per-book odds and explode into rows
raw = pd.read_sql("""
    SELECT id, game_date, home_team, away_team, odds_json
    FROM predictions
    WHERE odds_json IS NOT NULL
""", con)
raw['game_date'] = pd.to_datetime(raw['game_date'])

records = []
for _, row in raw.iterrows():
    odds = json.loads(row['odds_json'])
    for book, prices in odds.items():
        for team, line in prices.items():
            records.append({
                'game_id':   row['id'],
                'game_date': row['game_date'],
                'home_team': row['home_team'],
                'away_team': row['away_team'],
                'book':      book,
                'team':      team,
                'odds':      line,
                'is_home':   int(team == row['home_team']),
            })

odds_df = pd.DataFrame(records)
print(f'{len(odds_df):,} odds records  |  {odds_df["book"].nunique()} books  |  {odds_df["game_id"].nunique()} games')
odds_df.head()

In [ ]:
# Implied probability by book
def implied_prob(odds):
    return abs(odds) / (abs(odds) + 100) if odds < 0 else 100 / (odds + 100)

odds_df['implied_prob'] = odds_df['odds'].apply(implied_prob)

book_summary = odds_df.groupby('book').agg(
    lines=('odds', 'count'),
    avg_implied_prob=('implied_prob', 'mean'),
    avg_home_implied=('implied_prob', lambda x: x[odds_df.loc[x.index, 'is_home'] == 1].mean()),
).round(3)
book_summary

In [ ]:
# Line spread across books for a single game — pick any game_id
sample_id = df.sample(1)['id'].values[0]
sample_game = df[df['id'] == sample_id][['game_date','home_team','away_team']].iloc[0]
print(f"{sample_game['game_date'].date()}  {sample_game['away_team']} @ {sample_game['home_team']}")

odds_df[odds_df['game_id'] == sample_id][['book','team','odds','implied_prob']].sort_values(['team','book'])

## 7. Value picks — edge distribution

In [ ]:
# Compute edge for every game (model prob minus implied prob at best odds)
edge_df = df.copy()
edge_df['implied_prob'] = edge_df['pred_odds'].apply(
    lambda x: implied_prob(x) if pd.notna(x) else np.nan
)
edge_df['edge'] = edge_df['pred_prob'] - edge_df['implied_prob']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Edge distribution
axes[0].hist(edge_df['edge'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(0.05, color='green', lw=1.5, ls='--', label='5% threshold')
axes[0].set_xlabel('Model edge over implied prob')
axes[0].set_title('Edge distribution')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].legend()

# Accuracy by edge bucket (resolved only)
re = edge_df[edge_df['resolved'] == 1].dropna(subset=['edge'])
re['edge_bucket'] = pd.cut(re['edge'], bins=[-1, 0, 0.05, 0.1, 0.15, 0.2, 1],
                           labels=['<0%', '0-5%', '5-10%', '10-15%', '15-20%', '>20%'])
acc_by_edge = re.groupby('edge_bucket', observed=True)['correct'].agg(['mean','count'])
acc_by_edge.columns = ['accuracy', 'games']
axes[1].bar(acc_by_edge.index, acc_by_edge['accuracy'], color='steelblue')
axes[1].axhline(0.5, color='red', lw=1, ls='--')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title('Accuracy by edge bucket')
for i, (idx, row) in enumerate(acc_by_edge.iterrows()):
    axes[1].text(i, row['accuracy'] + 0.005, f"n={int(row['games'])}", ha='center', fontsize=8)

plt.tight_layout()
plt.show()
acc_by_edge

## 8. Team-level breakdown

In [ ]:
# How often each team was predicted to win, and actual accuracy
team_stats = resolved.groupby('predicted_winner').agg(
    predicted=('correct', 'count'),
    correct=('correct', 'sum'),
).assign(accuracy=lambda x: (x['correct'] / x['predicted']).round(3))
.sort_values('accuracy', ascending=False)

team_stats

In [ ]:
# Custom query — edit freely
pd.read_sql("""
    SELECT game_date, home_team, away_team,
           predicted_winner, home_win_prob, away_win_prob,
           actual_winner,
           CASE WHEN predicted_winner = actual_winner THEN 1 ELSE 0 END AS correct,
           home_best_odds, away_best_odds
    FROM predictions
    WHERE resolved = 1
      AND is_value_pick = 1
    ORDER BY game_date DESC
    LIMIT 20
""", con)

In [ ]:
con.close()
print('Done')